# GitPython - Python Git Library

GitPython is a Python library used to interact with Git repositories. It provides a high-level interface for Git operations including repository management, commits, branches, and more.

## Example 1: Repository Initialization and Basic Operations

In [ ]:
import git
import tempfile
import os

# Create a temporary directory for our test repository
with tempfile.TemporaryDirectory() as temp_dir:
    repo_path = os.path.join(temp_dir, "test_repo")
    
    # Initialize a new Git repository
    repo = git.Repo.init(repo_path)
    print(f"Repository initialized at: {repo_path}")
    print(f"Repository working directory: {repo.working_dir}")
    print(f"Repository git directory: {repo.git_dir}")
    print(f"Repository is bare: {repo.bare}")
    
    # Create a test file
    test_file = os.path.join(repo_path, "test.txt")
    with open(test_file, "w") as f:
        f.write("Hello, GitPython!")
    
    # Check repository status
    print(f"\nRepository status:")
    print(f"Is dirty (has changes): {repo.is_dirty()}")
    print(f"Untracked files: {repo.untracked_files}")
    
    # Add file to staging area
    repo.index.add([test_file])
    print(f"After adding file - Is dirty: {repo.is_dirty()}")

## Example 2: Working with Git Index and Commits

In [ ]:
import git
import tempfile
import os

with tempfile.TemporaryDirectory() as temp_dir:
    repo = git.Repo.init(temp_dir)
    
    # Configure user for commits
    with repo.config_writer() as config:
        config.set_value("user", "name", "Test User")
        config.set_value("user", "email", "test@example.com")
    
    # Create and add a file
    test_file = os.path.join(temp_dir, "example.py")
    with open(test_file, "w") as f:
        f.write("# Example Python file\nprint('Hello from GitPython!')")
    
    # Stage the file
    repo.index.add([test_file])
    
    # Create initial commit
    commit = repo.index.commit("Initial commit with example file")
    print(f"Created commit: {commit.hexsha[:8]}")
    print(f"Commit message: {commit.message.strip()}")
    print(f"Commit author: {commit.author}")
    print(f"Commit date: {commit.committed_datetime}")
    
    # Check commit stats
    print(f"\nCommit statistics:")
    print(f"Total commits: {len(list(repo.iter_commits()))}")
    print(f"Active branch: {repo.active_branch.name}")

## Example 3: Branch Operations and References

In [ ]:
import git
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    repo = git.Repo.init(temp_dir)
    
    # Configure user
    with repo.config_writer() as config:
        config.set_value("user", "name", "Test User")
        config.set_value("user", "email", "test@example.com")
    
    # Create initial commit
    with open(f"{temp_dir}/README.md", "w") as f:
        f.write("# Test Repository")
    repo.index.add(["README.md"])
    initial_commit = repo.index.commit("Initial commit")
    
    # List all branches
    print("Current branches:")
    for branch in repo.branches:
        print(f"  {branch.name} ({'active' if branch == repo.active_branch else 'inactive'})")
    
    # Create a new branch
    new_branch = repo.create_head("feature-branch")
    print(f"\nCreated new branch: {new_branch.name}")
    
    # Switch to new branch
    new_branch.checkout()
    print(f"Active branch is now: {repo.active_branch.name}")
    
    # List all references
    print(f"\nAll references:")
    for ref in repo.refs:
        print(f"  {ref.name}: {ref.commit.hexsha[:8]}")

## Example 4: Remote Repository and Git Commands

In [ ]:
import git
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    repo = git.Repo.init(temp_dir)
    
    # Configure user
    with repo.config_writer() as config:
        config.set_value("user", "name", "Test User")
        config.set_value("user", "email", "test@example.com")
    
    # Test Git command interface
    print("Git version information:")
    try:
        git_version = repo.git.version()
        print(f"Git version: {git_version}")
    except Exception as e:
        print(f"Could not get Git version: {e}")
        raise
    
    # Execute various Git commands
    print("\nTesting Git commands:")
    
    # Status command
    status_output = repo.git.status("--porcelain")
    print(f"Git status output: '{status_output}' (empty = clean)")
    
    # Log command (should be empty initially)
    try:
        log_output = repo.git.log("--oneline", "--max-count=5")
        print(f"Recent commits: {log_output}")
    except git.exc.GitCommandError:
        print("No commits yet in repository")
    
    # Show current branch
    try:
        current_branch = repo.git.branch("--show-current")
        print(f"Current branch: {current_branch}")
    except:
        print("Could not determine current branch")
        raise

## Example 5: Error Handling and Repository Validation

In [ ]:
import git
import tempfile
import os

print("Testing GitPython error handling:")

# Test invalid repository path
# A path that does not exist raises NoSuchPathError; InvalidGitRepositoryError is for an existing
# directory that is not a repository (next test).
try:
    invalid_repo = git.Repo("/invalid/path/to/repo")
except git.exc.NoSuchPathError as e:
    assert str(e) == "/invalid/path/to/repo", e
    print(f"✓ Correctly rejected a missing path ({type(e).__name__})")
else:
    raise AssertionError("expected git.exc.NoSuchPathError for a path that does not exist")

# Test non-git directory
with tempfile.TemporaryDirectory() as temp_dir:
    try:
        non_git_repo = git.Repo(temp_dir)
    except git.exc.InvalidGitRepositoryError as e:
        assert str(e) == temp_dir, e
        print("✓ Correctly identified non-git directory")
    else:
        raise AssertionError(f"git.Repo({temp_dir!r}) opened a directory that is not a repository")

# Test repository validation
with tempfile.TemporaryDirectory() as temp_dir:
    # Create a proper git repository
    repo = git.Repo.init(temp_dir)
    
    # Validate repository
    print(f"\nRepository validation:")
    print(f"Is valid git repository: {True}")  # If we got here, it's valid
    print(f"Repository path exists: {os.path.exists(repo.git_dir)}")
    print(f"Objects directory exists: {os.path.exists(os.path.join(repo.git_dir, 'objects'))}")
    print(f"Refs directory exists: {os.path.exists(os.path.join(repo.git_dir, 'refs'))}")
    
    # Test repository properties
    print(f"Working tree: {repo.working_tree_dir}")
    print(f"Common dir: {repo.common_dir}")

print("\nGitPython error handling tests completed!")